# Simple CUDA Metrics Demo (Colab)

Goal: show a red-teaming metrics workload where the GPU can beat the CPU once the input is large enough.

We start from the small synthetic CSV, expand it into a benchmark-sized dataset, and then run the same compute-heavy scoring pass on CPU and GPU.

We time only the compute stage so the comparison highlights parallel execution, not file parsing.

Class script (60s):
1. Cell 2 confirms the GPU runtime is available.
2. Cell 3 expands the red-team metrics into benchmark_metrics.csv.
3. Cells 4 and 5 write the CPU and CUDA source code.
4. Cell 6 compiles and runs both binaries.
5. Cell 7 checks that the outputs match.
6. Cell 8 reports the compute-time speedup.

In [7]:
from pathlib import Path
import csv
import shutil
import subprocess


def run_tool(command):
    completed = subprocess.run(command, shell=True, capture_output=True, text=True)
    if completed.stdout:
        print(completed.stdout, end='')
    if completed.stderr:
        print(completed.stderr, end='')
    return completed.returncode == 0


nvidia_smi = shutil.which('nvidia-smi')
if nvidia_smi is None:
    print('nvidia-smi not available in this environment. Run in Colab with a GPU runtime to check CUDA hardware.')
else:
    print('Checking GPU runtime...')
    run_tool('nvidia-smi')

print()
print('Checking nvcc...')
nvcc = shutil.which('nvcc')
if nvcc is None:
    print('nvcc not available in this environment. Run in Colab with a GPU runtime to compile the CUDA binary.')
else:
    run_tool('nvcc --version')

nvidia-smi not available in this environment. Run in Colab with a GPU runtime to check CUDA hardware.

Checking nvcc...
nvcc not available in this environment. Run in Colab with a GPU runtime to compile the CUDA binary.


In [2]:
from pathlib import Path
import csv

source_csv = Path('synthetic_metrics.csv')
benchmark_csv = Path('benchmark_metrics.csv')
if not source_csv.exists():
    raise FileNotFoundError('synthetic_metrics.csv not found. Upload this file into the same Colab working directory.')

with source_csv.open(newline='', encoding='utf-8') as handle:
    rows = list(csv.reader(handle))

header = rows[0]
data_rows = rows[1:]
repeat_factor = 5000
expanded_rows = [header]
for _ in range(repeat_factor):
    expanded_rows.extend(data_rows)

with benchmark_csv.open('w', newline='', encoding='utf-8') as handle:
    writer = csv.writer(handle)
    writer.writerows(expanded_rows)

print('Source rows:', len(data_rows))
print('Benchmark rows:', len(expanded_rows) - 1)
print('Benchmark file:', benchmark_csv)
print('First benchmark rows:')
for row in expanded_rows[:6]:
    print(row)

Source rows: 32
Benchmark rows: 160000
Benchmark file: benchmark_metrics.csv
First benchmark rows:
['harm_category_index', 'final_score', 'iterations', 'success_flag']
['0', '0.12', '1', '0']
['1', '0.91', '3', '0']
['2', '0.98', '4', '1']
['3', '0.45', '2', '0']
['4', '0.77', '3', '0']


In [3]:
%%writefile cpu_metrics.cpp
#include <algorithm>
#include <chrono>
#include <cmath>
#include <fstream>
#include <iomanip>
#include <iostream>
#include <sstream>
#include <string>
#include <vector>

struct Row {
    float score;
    int iterations;
    int success;
};

static float harden_score(float score, int iterations, int success) {
    float value = score;
    const float drift = 1.0005f + 0.0001f * static_cast<float>(iterations);
    const float bias = success ? 0.00025f : -0.00015f;
    for (int pass = 0; pass < 256; ++pass) {
        value = value * drift + bias;
        value -= std::floor(value);
    }
    return value;
}

int main(int argc, char** argv) {
    if (argc < 2) {
        std::cerr << "Usage: ./cpu_metrics <csv_file>\n";
        return 1;
    }

    const auto load_start = std::chrono::steady_clock::now();
    std::ifstream file(argv[1]);
    if (!file.is_open()) {
        std::cerr << "Could not open file: " << argv[1] << "\n";
        return 1;
    }

    std::string line;
    std::getline(file, line); // skip header

    std::vector<Row> rows;
    while (std::getline(file, line)) {
        if (line.empty()) continue;
        std::stringstream ss(line);
        std::string token;

        std::getline(ss, token, ','); // harm_category_index
        std::getline(ss, token, ','); // final_score
        float score = std::stof(token);
        std::getline(ss, token, ','); // iterations
        int iterations = std::stoi(token);
        std::getline(ss, token, ','); // success_flag
        int success = std::stoi(token);

        rows.push_back({score, iterations, success});
    }
    const auto load_end = std::chrono::steady_clock::now();

    if (rows.empty()) {
        std::cerr << "No data rows found\n";
        return 1;
    }

    const auto compute_start = std::chrono::steady_clock::now();
    int total_rows = 0;
    int success_count = 0;
    float score_sum = 0.0f;
    float max_score = 0.0f;

    for (const Row& row : rows) {
        const float hardened = harden_score(row.score, row.iterations, row.success);
        total_rows += 1;
        success_count += row.success;
        score_sum += hardened;
        max_score = std::max(max_score, hardened);
    }
    const auto compute_end = std::chrono::steady_clock::now();
    const auto total_end = std::chrono::steady_clock::now();

    const double load_seconds = std::chrono::duration<double>(load_end - load_start).count();
    const double compute_seconds = std::chrono::duration<double>(compute_end - compute_start).count();
    const double total_seconds = std::chrono::duration<double>(total_end - load_start).count();

    std::cout << std::fixed << std::setprecision(6);
    std::cout << "load_seconds=" << load_seconds << "\n";
    std::cout << "compute_seconds=" << compute_seconds << "\n";
    std::cout << "total_seconds=" << total_seconds << "\n";
    std::cout << "total_rows=" << total_rows << "\n";
    std::cout << "success_count=" << success_count << "\n";
    std::cout << "score_sum=" << score_sum << "\n";
    std::cout << "max_score=" << max_score << "\n";
    return 0;
}

Writing cpu_metrics.cpp


In [4]:
%%writefile gpu_metrics.cu
#include <cuda_runtime.h>

#include <algorithm>
#include <chrono>
#include <cmath>
#include <cstdlib>
#include <fstream>
#include <iomanip>
#include <iostream>
#include <sstream>
#include <string>
#include <vector>

#define CUDA_CHECK(call) do { \
  cudaError_t err = (call); \
  if (err != cudaSuccess) { \
    std::cerr << "CUDA error: " << cudaGetErrorString(err) << "\n"; \
    std::exit(1); \
  } \
} while (0)

struct Row {
  float score;
  int iterations;
  int success;
};

__device__ float atomicMaxFloat(float* address, float val) {
  int* address_as_i = reinterpret_cast<int*>(address);
  int old = *address_as_i, assumed;
  while (__int_as_float(old) < val) {
    assumed = old;
    old = atomicCAS(address_as_i, assumed, __float_as_int(val));
    if (assumed == old) break;
  }
  return __int_as_float(old);
}

__device__ float harden_score(float score, int iterations, int success) {
  float value = score;
  const float drift = 1.0005f + 0.0001f * static_cast<float>(iterations);
  const float bias = success ? 0.00025f : -0.00015f;
  for (int pass = 0; pass < 256; ++pass) {
    value = value * drift + bias;
    value -= floorf(value);
  }
  return value;
}

__global__ void metrics_kernel(const float* scores, const int* iterations, const int* successes, int n,
                               int* total_rows, int* success_count,
                               float* score_sum, float* max_score) {
  int i = blockIdx.x * blockDim.x + threadIdx.x;
  if (i >= n) return;

  const float hardened = harden_score(scores[i], iterations[i], successes[i]);
  atomicAdd(total_rows, 1);
  atomicAdd(success_count, successes[i]);
  atomicAdd(score_sum, hardened);
  atomicMaxFloat(max_score, hardened);
}

int main(int argc, char** argv) {
  if (argc < 2) {
    std::cerr << "Usage: ./gpu_metrics <csv_file>\n";
    return 1;
  }

  const auto load_start = std::chrono::steady_clock::now();
  std::ifstream file(argv[1]);
  if (!file.is_open()) {
    std::cerr << "Could not open file: " << argv[1] << "\n";
    return 1;
  }

  std::string line;
  std::getline(file, line); // skip header

  std::vector<float> h_scores;
  std::vector<int> h_iterations;
  std::vector<int> h_successes;

  while (std::getline(file, line)) {
    if (line.empty()) continue;
    std::stringstream ss(line);
    std::string token;

    std::getline(ss, token, ','); // harm_category_index
    std::getline(ss, token, ','); // final_score
    float score = std::stof(token);
    std::getline(ss, token, ','); // iterations
    int iterations = std::stoi(token);
    std::getline(ss, token, ','); // success_flag
    int success = std::stoi(token);

    h_scores.push_back(score);
    h_iterations.push_back(iterations);
    h_successes.push_back(success);
  }
  const auto load_end = std::chrono::steady_clock::now();

  int n = static_cast<int>(h_scores.size());
  if (n == 0) {
    std::cerr << "No data rows found\n";
    return 1;
  }

  float* d_scores = nullptr;
  int* d_iterations = nullptr;
  int* d_successes = nullptr;
  int* d_total_rows = nullptr;
  int* d_success_count = nullptr;
  float* d_score_sum = nullptr;
  float* d_max_score = nullptr;

  CUDA_CHECK(cudaMalloc(&d_scores, n * sizeof(float)));
  CUDA_CHECK(cudaMalloc(&d_iterations, n * sizeof(int)));
  CUDA_CHECK(cudaMalloc(&d_successes, n * sizeof(int)));
  CUDA_CHECK(cudaMalloc(&d_total_rows, sizeof(int)));
  CUDA_CHECK(cudaMalloc(&d_success_count, sizeof(int)));
  CUDA_CHECK(cudaMalloc(&d_score_sum, sizeof(float)));
  CUDA_CHECK(cudaMalloc(&d_max_score, sizeof(float)));

  CUDA_CHECK(cudaMemcpy(d_scores, h_scores.data(), n * sizeof(float), cudaMemcpyHostToDevice));
  CUDA_CHECK(cudaMemcpy(d_iterations, h_iterations.data(), n * sizeof(int), cudaMemcpyHostToDevice));
  CUDA_CHECK(cudaMemcpy(d_successes, h_successes.data(), n * sizeof(int), cudaMemcpyHostToDevice));

  CUDA_CHECK(cudaMemset(d_total_rows, 0, sizeof(int)));
  CUDA_CHECK(cudaMemset(d_success_count, 0, sizeof(int)));
  CUDA_CHECK(cudaMemset(d_score_sum, 0, sizeof(float)));
  CUDA_CHECK(cudaMemset(d_max_score, 0, sizeof(float)));

  cudaEvent_t kernel_start = nullptr;
  cudaEvent_t kernel_stop = nullptr;
  CUDA_CHECK(cudaEventCreate(&kernel_start));
  CUDA_CHECK(cudaEventCreate(&kernel_stop));

  const int block_size = 256;
  const int grid_size = (n + block_size - 1) / block_size;
  CUDA_CHECK(cudaEventRecord(kernel_start));
  metrics_kernel<<<grid_size, block_size>>>(d_scores, d_iterations, d_successes, n, d_total_rows, d_success_count, d_score_sum, d_max_score);
  CUDA_CHECK(cudaGetLastError());
  CUDA_CHECK(cudaEventRecord(kernel_stop));
  CUDA_CHECK(cudaEventSynchronize(kernel_stop));

  float kernel_ms = 0.0f;
  CUDA_CHECK(cudaEventElapsedTime(&kernel_ms, kernel_start, kernel_stop));

  int total_rows = 0;
  int success_count = 0;
  float score_sum = 0.0f;
  float max_score = 0.0f;

  CUDA_CHECK(cudaMemcpy(&total_rows, d_total_rows, sizeof(int), cudaMemcpyDeviceToHost));
  CUDA_CHECK(cudaMemcpy(&success_count, d_success_count, sizeof(int), cudaMemcpyDeviceToHost));
  CUDA_CHECK(cudaMemcpy(&score_sum, d_score_sum, sizeof(float), cudaMemcpyDeviceToHost));
  CUDA_CHECK(cudaMemcpy(&max_score, d_max_score, sizeof(float), cudaMemcpyDeviceToHost));

  CUDA_CHECK(cudaEventDestroy(kernel_start));
  CUDA_CHECK(cudaEventDestroy(kernel_stop));
  CUDA_CHECK(cudaFree(d_scores));
  CUDA_CHECK(cudaFree(d_iterations));
  CUDA_CHECK(cudaFree(d_successes));
  CUDA_CHECK(cudaFree(d_total_rows));
  CUDA_CHECK(cudaFree(d_success_count));
  CUDA_CHECK(cudaFree(d_score_sum));
  CUDA_CHECK(cudaFree(d_max_score));

  const auto total_end = std::chrono::steady_clock::now();
  const double load_seconds = std::chrono::duration<double>(load_end - load_start).count();
  const double compute_seconds = static_cast<double>(kernel_ms) / 1000.0;
  const double total_seconds = std::chrono::duration<double>(total_end - load_start).count();

  std::cout << std::fixed << std::setprecision(6);
  std::cout << "load_seconds=" << load_seconds << "\n";
  std::cout << "compute_seconds=" << compute_seconds << "\n";
  std::cout << "total_seconds=" << total_seconds << "\n";
  std::cout << "total_rows=" << total_rows << "\n";
  std::cout << "success_count=" << success_count << "\n";
  std::cout << "score_sum=" << score_sum << "\n";
  std::cout << "max_score=" << max_score << "\n";
  return 0;
}

Writing gpu_metrics.cu


In [8]:
import os
import shutil
import subprocess
from pathlib import Path


def run_command(command):
    completed = subprocess.run(command, shell=True, capture_output=True, text=True)
    if completed.stdout:
        print(completed.stdout, end='')
    if completed.stderr:
        print(completed.stderr, end='')
    if completed.returncode != 0:
        raise subprocess.CalledProcessError(
            completed.returncode,
            command,
            output=completed.stdout,
            stderr=completed.stderr,
        )
    return completed.stdout


def executable_name(base_name):
    return f'{base_name}.exe' if os.name == 'nt' else f'./{base_name}'


benchmark_csv = 'benchmark_metrics.csv'
cpu_exe = executable_name('cpu_metrics')
gpu_exe = executable_name('gpu_metrics')
nvcc_available = shutil.which('nvcc') is not None

print('Compiling CPU program...')
run_command(f'g++ -O3 -std=c++17 cpu_metrics.cpp -o {cpu_exe}')

if nvcc_available:
    print('Compiling CUDA program...')
    run_command(f'nvcc -O3 -std=c++17 gpu_metrics.cu -o {gpu_exe}')
else:
    print('nvcc not available here; skipping CUDA compile. Run this notebook in Colab with a GPU runtime for the full CPU vs GPU demo.')

print()
print('CPU output:')
cpu_output = run_command(f'{cpu_exe} {benchmark_csv}')
Path('cpu_output.txt').write_text(cpu_output, encoding='utf-8')

if nvcc_available:
    print()
    print('GPU output:')
    gpu_output = run_command(f'{gpu_exe} {benchmark_csv}')
    Path('gpu_output.txt').write_text(gpu_output, encoding='utf-8')

Compiling CPU program...
nvcc not available here; skipping CUDA compile. Run this notebook in Colab with a GPU runtime for the full CPU vs GPU demo.

CPU output:
load_seconds=0.694331
compute_seconds=1.060566
total_seconds=1.754897
total_rows=160000
success_count=50000
score_sum=60911.117188
max_score=0.999892


In [9]:
from pathlib import Path


def parse_metrics(text):
    out = {}
    for line in text.strip().splitlines():
        if '=' not in line:
            continue
        key, value = line.split('=', 1)
        out[key.strip()] = float(value.strip())
    return out


cpu_output_path = Path('cpu_output.txt')
gpu_output_path = Path('gpu_output.txt')
if not cpu_output_path.exists() or not gpu_output_path.exists():
    print('Run Cell 6 in a GPU-enabled runtime to generate both output files before checking parity.')
else:
    cpu_metrics = parse_metrics(cpu_output_path.read_text(encoding='utf-8'))
    gpu_metrics = parse_metrics(gpu_output_path.read_text(encoding='utf-8'))

    eps = 1e-2
    checks = {
        'total_rows': int(cpu_metrics['total_rows']) == int(gpu_metrics['total_rows']),
        'success_count': int(cpu_metrics['success_count']) == int(gpu_metrics['success_count']),
        'score_sum': abs(cpu_metrics['score_sum'] - gpu_metrics['score_sum']) <= eps,
        'max_score': abs(cpu_metrics['max_score'] - gpu_metrics['max_score']) <= eps,
    }

    for name, ok in checks.items():
        print(f'{name}:', 'PASS' if ok else 'FAIL')

    all_ok = all(checks.values())
    print('\nParity result:', 'PASS' if all_ok else 'FAIL')
    if not all_ok:
        raise AssertionError('CPU and GPU results do not match.')

Run Cell 6 in a GPU-enabled runtime to generate both output files before checking parity.


In [10]:
import os
import shutil
import subprocess
from pathlib import Path


def run_command(command):
    completed = subprocess.run(command, shell=True, capture_output=True, text=True)
    if completed.stdout:
        print(completed.stdout, end='')
    if completed.stderr:
        print(completed.stderr, end='')
    if completed.returncode != 0:
        raise subprocess.CalledProcessError(
            completed.returncode,
            command,
            output=completed.stdout,
            stderr=completed.stderr,
        )
    return completed.stdout


def executable_name(base_name):
    return f'{base_name}.exe' if os.name == 'nt' else f'./{base_name}'


def parse_compute_seconds(output):
    for line in output.strip().splitlines():
        if line.startswith('compute_seconds='):
            return float(line.split('=', 1)[1])
    raise ValueError('compute_seconds not found in program output')


cpu_output_path = Path('cpu_output.txt')
gpu_output_path = Path('gpu_output.txt')
if not cpu_output_path.exists() or not gpu_output_path.exists():
    print('Run Cell 6 in a GPU-enabled runtime to generate both output files before measuring speedup.')
else:
    cpu_exe = executable_name('cpu_metrics')
    gpu_exe = executable_name('gpu_metrics')
    nvcc_available = shutil.which('nvcc') is not None

    print(f'Benchmark rows: {sum(1 for _ in Path("benchmark_metrics.csv").open(encoding="utf-8")) - 1}')
    print()

    cpu_times = []
    gpu_times = []

    for _ in range(5):
        cpu_times.append(parse_compute_seconds(run_command(f'{cpu_exe} benchmark_metrics.csv')))

    if nvcc_available:
        for _ in range(5):
            gpu_times.append(parse_compute_seconds(run_command(f'{gpu_exe} benchmark_metrics.csv')))
    else:
        print('nvcc is not available here, so the GPU timing loop is skipped.')

    cpu_avg = sum(cpu_times) / len(cpu_times)
    print(f'CPU compute times: {" ".join(f"{value:.6f}" for value in cpu_times)}')
    print(f'Average CPU compute time: {cpu_avg:.6f} s')

    if gpu_times:
        gpu_avg = sum(gpu_times) / len(gpu_times)
        speedup = cpu_avg / gpu_avg if gpu_avg > 0 else float("inf")
        print(f'GPU compute times: {" ".join(f"{value:.6f}" for value in gpu_times)}')
        print(f'Average GPU compute time: {gpu_avg:.6f} s')
        print(f'Compute speedup (CPU/GPU): {speedup:.2f}x')
    else:
        print('Run this notebook in a GPU runtime to measure GPU compute time and speedup.')

Run Cell 6 in a GPU-enabled runtime to generate both output files before measuring speedup.
